# Les décorateurs

In [ ]:
def sandwich():
    print("/TTTTTTTT\\")
    print("  Jambon ")
    print("  Beurre ")
    print("\\________/")

In [ ]:
sandwich()

## Principe

In [ ]:
def sandwich(content):
    def wrapper(*args, **kwargs):
        print("/TTTTTTTT\\")
        price = content(*args, **kwargs)
        print("\\________/")
        return price + 1

    return wrapper

In [ ]:
@sandwich
def parisien():
    print("  Jambon ")
    print("  Beurre ")
    return 5

@sandwich
def lyonnais():
    print("  Rosette ")
    print("  Beurre ")
    return 4.5

@sandwich
def kebab(salade_tomates_oignons=True):
    if salade_tomates_oignons:
        print("  salade, tomates, oignons ")
    print("  Viande ")
    return 6


In [ ]:
kebab(salade_tomates_oignons=False)

In [ ]:
print(parisien())

In [ ]:
lyonnais()

### Décorteurs paramétrés

In [24]:
def hsandwich(_func=None, *, hot=False):
    def inner_deco(content):
        def wrapper(*args, **kwargs):
            print("/TTTTTTTT\\")
            price = content(*args, **kwargs)
            print("\\________/")
            print(f"{hot}")
            return price + 1

        return wrapper

    if _func is None:
        return inner_deco
    else:
        return inner_deco(_func)

@hsandwich
def parisien():
    print("  Jambon ")
    print("  Beurre ")
    return 5


In [25]:
parisien()

/TTTTTTTT\
  Jambon 
  Beurre 
\________/
False


6

## Exercices

### Premier exercice

In [ ]:
import time

def time_it(func):
    def wrapper(*args, **kwargs):
        start = time.time()

        value = func(*args, **kwargs)

        end = time.time()
        print(f"{end - start:.6f} secondes se sont écoulées")
        return value

    return wrapper

@time_it
def job(nb_runs=1_000_000):
    y = 0
    for x in range(nb_runs):
        y = x ** 2
    return y


In [ ]:
job(10_000_000)

## Second exercice

In [ ]:
def count_calls(func):
    count = 0
    def inner(*args, **kwargs):
        nonlocal count
        value = func(*args, **kwargs)
        count += 1
        return value

    def inner_get_count():
        return count

    inner.nbcalls = inner_get_count

    return inner

@count_calls
def some_func():
    print("func called")

In [ ]:
print(some_func.nbcalls())
some_func()
some_func()
print(some_func.nbcalls())

In [ ]:
@count_calls
@time_it
def job(nb_runs=1_000_000):
    y = 0
    for x in range(nb_runs):
        y = x ** 2
    return y


In [ ]:
print(job.nbcalls())
print(job(10_000_000))
print(job(1_000_000))
print(job.nbcalls())

## Troisième exercice

In [ ]:
import time
from collections import OrderedDict

def cached_call(func):

    cached_data = OrderedDict()

    def inner(value:int):
        if value in cached_data:
            cached_data.move_to_end(value)
            return cached_data[value]

        result = func(value)

        cached_data[value] = result
        if len(cached_data) > 3:
            print("poped", cached_data.popitem(last=False))
        return result

    return inner

@time_it
@cached_call
def long_call(value:int):
    time.sleep(2)
    return value**2

In [ ]:
long_call(5)
long_call(10)
long_call(5)
long_call(20)
long_call(30)

In [ ]:
long_call(5)

## Quatrième exercice

In [1]:
notifications = []

def register(_func=None, *, level:int=0):
    def register_func(func):
        notifications.append((level, func))
        return func

    if _func is None:
        return register_func
    else:
        return register_func(_func)

@register(level=2)
def notify_mail():
    print("notification on mail")

@register
def notify_sms():
    print("notification on message")

@register(level=1)
def notify_push():
    print("notification on push service")

def send_notifications(level=0):
    for notif_level, notification in notifications:
        if notif_level >= level:
            notification()

In [4]:
send_notifications(2)

notification on mail
